In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_DIR = Path(
    '/content/drive/MyDrive/CaReFlow/checkpoints'
)
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

print("Google Drive 保存目录：", DRIVE_DIR)

In [ ]:
%%bash
set -e
if git -C /content/CaReFlow rev-parse --is-inside-work-tree >/dev/null 2>&1; then
    git -C /content/CaReFlow pull --ff-only origin main
else
    git clone https://github.com/Scream9371/CaReFlow.git /content/CaReFlow
fi


In [ ]:
%%bash
set -e
cd /content/CaReFlow/datasets

if [[ ! -s mosi.pkl || ! -s mosei.pkl ]]; then
    python -m pip install -q gdown
fi
if [[ ! -s mosi.pkl ]]; then
    gdown 'https://drive.google.com/uc?id=12HbavGOtoVCqicvSYWl3zImli5Jz0Nou' -O mosi.pkl
fi
if [[ ! -s mosei.pkl ]]; then
    gdown 'https://drive.google.com/uc?id=1VJhSc2TGrPU8zJSVTYwn5kfuG47VaNQ3' -O mosei.pkl
fi

for dataset in mosi.pkl mosei.pkl; do
    test -s "$dataset" || { echo "数据集缺失：$dataset" >&2; exit 1; }
done
ls -lh mosi.pkl mosei.pkl


In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download

model_dir = Path('/content/CaReFlow/microsoft/deberta-v3-base')
required = [model_dir / 'config.json', model_dir / 'spm.model']
weights = [model_dir / 'model.safetensors', model_dir / 'pytorch_model.bin']

if all(path.is_file() and path.stat().st_size > 0 for path in required) and any(
    path.is_file() and path.stat().st_size > 0 for path in weights
):
    print('DeBERTa 预训练模型已存在，跳过下载。')
else:
    print('下载 DeBERTa 预训练模型...')
    snapshot_download(
        repo_id='microsoft/deberta-v3-base',
        local_dir=str(model_dir),
        ignore_patterns=['*.msgpack', '*.h5', '*.ot'],
    )
    assert all(path.is_file() and path.stat().st_size > 0 for path in required)
    assert any(path.is_file() and path.stat().st_size > 0 for path in weights)
    print('下载完成。')


In [ ]:
!ls -lh /content/CaReFlow/microsoft/deberta-v3-base

In [ ]:
!python -m pip install "transformers==4.57.6"
!python -c "import torch, transformers; print('PyTorch:', torch.__version__); print('Transformers:', transformers.__version__); print('CUDA:', torch.cuda.is_available())"

In [ ]:
%%bash
set -e
cd /content/CaReFlow
python -u train_reflow_new.py \
    --dataset mosi \
    --n_epochs 100 \
    --train_batch_size 32 \
    --learning_rate 10e-6 \
    --ratio 4 \
    --dropout_prob 0.5 \
    --inter_dim 150 \
    --share_dim 100 \
    --transformer_layer 3 \
    --step_size 2 \
    --loss_b_ratio 0.1 \
    --loss_f_ratio 0.2 \
    --eps 1e-3 \
    --save_model \
    --output_dir /content/CaReFlow/checkpoints
cp /content/CaReFlow/checkpoints/careflow_mosi_best.pt /content/drive/MyDrive/CaReFlow/checkpoints/


In [ ]:
%%bash
set -e
cd /content/CaReFlow
python -u train_reflow_new.py \
    --dataset mosei \
    --n_epochs 100 \
    --train_batch_size 32 \
    --learning_rate 10e-6 \
    --ratio 4 \
    --dropout_prob 0.5 \
    --inter_dim 150 \
    --share_dim 100 \
    --transformer_layer 3 \
    --step_size 2 \
    --loss_b_ratio 0.1 \
    --loss_f_ratio 0.2 \
    --eps 1e-3 \
    --save_model \
    --output_dir /content/CaReFlow/checkpoints
cp /content/CaReFlow/checkpoints/careflow_mosei_best.pt /content/drive/MyDrive/CaReFlow/checkpoints/
